# Value-code explorer (sampled networks)
Author: patrick.mccarthy@dpag.ox.ac.uk

Hypothetical codes an RNN critic could use to represent **stimulus identity** and **value** (0% / 50% / 100%),
each built as an actual small sampled ReLU network. All code lives in `scripts/02_10_26_value_code_schematics.py`.

**Generative model.** $N$ units are split into $G$ groups ($G=3$ subgroups or $G=1$ single population) in proportion
to `sizes`. For unit $i$ in group $g$:

| quantity | distribution |
|---|---|
| input weight from stimulus $s$ | $w^{in}_{is} \sim \mathcal N(\mu^{in}_{gs},\ \sigma_{in}^2)$ |
| readout weight | $w^{out}_i \sim \mathcal N(\mu^{out}_g,\ \sigma_{out}^2)$ |
| bias / threshold | $b_i \sim \mathcal N(\mu_b,\ \sigma_b^2)$ |
| recurrent weight (within group only) | $W^{rec}_{ij} \sim \mathcal N(\mu^{rec}_g / n_g,\ \sigma_{rec}^2 / n_g)$ |

So $\mu^{in}$ is a groups × stimuli matrix of mean input strengths (shift a mean to change strength; off-diagonal
entries make groups mixed-selective), and $\mu^{rec}_g$ is the group's recurrent gain (rows sum to ≈ $\mu^{rec}_g$).

**Activity** to stimulus $s$ (one-hot input) is the steady state $r = \mathrm{ReLU}(w^{in}_{:,s} + W^{rec} r + b)$.
**Value output** is an actual readout, $\hat V(s) = w^{out}\cdot r(s) / N$. With `readout="fit"` the readout is learned
instead: the minimum-norm $w^{out}$ (plus output bias) that maps $r(s)$ exactly onto the values.

**What's plotted.**
* *Schematic*: group-level **means** of the sampled parameters. Node area ∝ # units; grey shading = the group's mean
  activity to its own stimulus (single population: bands = population mean under each stimulus); edge width ∝
  |mean weight|; loop = recurrent gain.
* *Heatmap*: simulated stimulus × unit activity, sorted like transfer `heatmap_seed` — by preferred stimulus
  (argmax), then by peak response; units whose peak is < 5% of the max go to the bottom.
* *Readout*: $\hat V$ per stimulus, and the R² of the best output gain + bias mapping it onto the values
  ("affine" = straight line with an offset; < 1 means a linear readout can't produce exact values).
* *Responder categories*: which stimuli each unit responds to ($r$ > `RESP_THRESH`), as in transfer.
* *Features*: the four yes/no columns, **inferred** from the parameters and simulated activity.

## Setup

In [ ]:
%matplotlib inline
import importlib.util
from dataclasses import replace
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

REPO_ROOT = Path.cwd().parent  # nb/ is one level below the repo root
_spec = importlib.util.spec_from_file_location(
    "vcs", REPO_ROOT / "scripts" / "02_10_26_value_code_schematics.py")
vcs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(vcs)

## 1. All presets
Module-level knobs are read at call time: set them on `vcs` and redraw. Individual presets can be modified with
`dataclasses.replace`. Set `SAVE_AS` to write png/pdf/svg to `figures/02_10_26_value_code_schematics/`.

In [ ]:
vcs.N_UNITS = 64
vcs.SEED = 0
vcs.BASELINE = 0.25       # floor on value-graded quantities in the presets
vcs.RESP_THRESH = 0.1
vcs.HEAT_VMAX = 1.2

codes = vcs.default_codes()
# e.g. noisier input weights everywhere:
# codes = [replace(c, sd_in=0.3) for c in codes]
SAVE_AS = None            # e.g. "value_code_solutions"
fig = vcs.make_figure(codes, SAVE_AS)
plt.show()

## 2. Do purely mixed-selective cells look like subgroups?
A single population (no subgroups), every unit gets input from every stimulus, with weights drawn independently
from $\mathcal N(\mu_s, \sigma_{in}^2)$ where only the **mean** differs by stimulus. Increase $\sigma_{in}$ and add a negative
threshold $\mu_b$ and compare with the subgroup codes above.

In [ ]:
mixed = replace(vcs.CODES[4], name="Pure mixed selectivity\n(graded input means)",
                note="no subgroups; sd_in = 1, threshold -0.5",
                sd_in=1.0, mu_b=-0.5)
fig = vcs.explore(mixed); plt.show()

**Sweep.** Large population (2000 units) so the fractions are stable: fraction of responsive units preferring each
stimulus, fraction responding to a single stimulus only, and the readout R².

In [ ]:
vcs.N_UNITS = 2000
rows = []
for sd in [0.1, 0.3, 0.6, 1.0]:
    for mb in [0.0, -0.5]:
        sim = replace(vcs.CODES[4], sd_in=sd, mu_b=mb).simulate()
        R = sim["R"]
        act = R.max(0) > vcs.RESP_THRESH
        pref = np.bincount(R[:, act].argmax(0), minlength=3) / max(act.sum(), 1)
        n = vcs.responder_counts(R)
        rows.append((sd, mb, *np.round(pref, 2), round(n[:3].sum() / n.sum(), 2),
                     round(vcs.readout_fit(sim["vhat"])[1], 4)))
vcs.N_UNITS = 64
print(f"{'sd_in':>5} {'mu_b':>5} | prefer 0% 50% 100% | single-stim only | readout R²")
for r in rows:
    print(f"{r[0]:>5} {r[1]:>5} |      {r[2]:.2f} {r[3]:.2f} {r[4]:.2f} |      {r[5]:.2f}        | {r[6]}")

**A test that separates them.** If responsiveness to each stimulus is independent across units (what random mixed
selectivity predicts), the responder-category counts should equal the product of the per-stimulus response
probabilities. True subgroups show an *excess* of single-stimulus responders and a *deficit* of multi-responders.
`vcs.independence_check(R)` returns (observed, expected-under-independence); the same test could be run on the
transfer responder-group counts.

In [ ]:
vcs.N_UNITS = 500
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
labels = [l.replace("\n", " ") for l in vcs.RESP_LABELS] + ["none"]
for ax, c in zip(axes, [vcs.CODES[1], mixed]):
    obs, exp = vcs.independence_check(c.simulate()["R"])
    x = np.arange(len(obs))
    ax.bar(x - 0.2, obs, 0.4, label="observed", color="0.3")
    ax.bar(x + 0.2, exp, 0.4, label="expected if independent", color="0.75")
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_title(c.name.replace("\n", " "), fontsize=10)
axes[0].set_ylabel("# units"); axes[0].legend(fontsize=8)
vcs.N_UNITS = 64
plt.tight_layout(); plt.show()

## 3. Interactive explorer
Pick a preset, then edit. `μ_in` is a groups × stimuli matrix (row = receiving group, column = stimulus); with
**subgroup structure** unticked there is one population and only the first `μ_in` row, `μ_out` and `μ_rec` entries are used.
Sliders update on release; changing preset redraws once.

In [ ]:
PRESETS = {c.name.replace("\n", " "): c for c in vcs.default_codes()}
G_NAMES = ["grp 0%", "grp 50%", "grp 100%"]
S = dict(continuous_update=False, readout_format=".2f", style={"description_width": "90px"})

def fs(desc, lo, hi, step=0.05, value=0.0):
    # each slider gets its OWN Layout (a shared one would hide/show all sliders together)
    return widgets.FloatSlider(value=value, min=lo, max=hi, step=step, description=desc,
                               layout=widgets.Layout(width="270px"), **S)

sizes = [fs(f"size {g}", 0.0, 1.5) for g in G_NAMES]
mu_in = [[fs(f"μ_in {s}→{g[4:]}", -1.0, 1.5) for s in vcs.STIM_NAMES] for g in G_NAMES]
mu_out = [fs(f"μ_out {g}", -1.0, 1.5) for g in G_NAMES]
mu_rec = [fs(f"μ_rec {g}", 0.0, 0.95, 0.01) for g in G_NAMES]
sd_in, sd_out, sd_rec = fs("σ_in", 0, 1.5, 0.01), fs("σ_out", 0, 1.5, 0.01), fs("σ_rec", 0, 0.5, 0.01)
mu_b, sd_b = fs("μ_b", -1.0, 1.0), fs("σ_b", 0, 1.0, 0.01)
preset = widgets.Dropdown(options=list(PRESETS), description="preset", layout=widgets.Layout(width="440px"))
subgroups = widgets.Checkbox(value=True, description="subgroup structure")
readout = widgets.Dropdown(options=["sampled", "fit"], description="readout", layout=widgets.Layout(width="200px"))
n_units = widgets.IntSlider(value=64, min=6, max=512, step=2, description="n units", continuous_update=False)
seed = widgets.IntSlider(value=0, min=0, max=50, description="seed", continuous_update=False)
thresh = fs("resp thresh", 0.0, 1.0, 0.01, 0.1)

all_w = (sizes + sum(mu_in, []) + mu_out + mu_rec +
         [sd_in, sd_out, sd_rec, mu_b, sd_b, readout, n_units, seed, thresh])
import io
out = widgets.Output()      # text messages (warnings) only
img = widgets.Image(format="png")   # the figure: replaced in place on every redraw, never stacked
_hold = False


def current_code():
    g3 = subgroups.value
    v = lambda ws: np.array([w.value for w in ws])
    M = np.array([[w.value for w in row] for row in mu_in])
    return vcs.NetCode(
        preset.value + " (edited)", "",
        sizes=v(sizes) if g3 else np.ones(1),
        mu_in=M if g3 else M[:1], sd_in=sd_in.value,
        mu_out=v(mu_out) if g3 else v(mu_out)[:1], sd_out=sd_out.value,
        mu_rec=v(mu_rec) if g3 else v(mu_rec)[:1], sd_rec=sd_rec.value,
        mu_b=mu_b.value, sd_b=sd_b.value, readout=readout.value,
        n_units=n_units.value, seed=seed.value)


def redraw(change=None):
    if _hold:
        return
    vcs.RESP_THRESH = thresh.value
    out.clear_output()
    with out:
        fig = vcs.explore(current_code(), wmax=1.5)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    plt.close(fig)
    img.value = buf.getvalue()


ROW0_LABELS = dict(  # slider labels for the single population vs subgroup modes
    sub=([f"μ_in {s}→{g[4:]}" for s in vcs.STIM_NAMES for g in G_NAMES[:1]], f"μ_out {G_NAMES[0]}", f"μ_rec {G_NAMES[0]}"),
    pop=([f"μ_in {s}→pop" for s in vcs.STIM_NAMES], "μ_out pop", "μ_rec pop"))


def set_mode(single):
    """Show only the population's sliders in single-population mode (rows for groups 50%/100%
    and the group sizes are hidden), and relabel the first row."""
    disp = "none" if single else None
    for w in sizes + sum(mu_in[1:], []) + mu_out[1:] + mu_rec[1:]:
        w.layout.display = disp
    size_box.layout.display = disp
    lab_in, lab_out, lab_rec = ROW0_LABELS["pop" if single else "sub"]
    for w, l in zip(mu_in[0], lab_in if single else [f"μ_in {s}→{G_NAMES[0][4:]}" for s in vcs.STIM_NAMES]):
        w.description = l
    mu_out[0].description, mu_rec[0].description = lab_out, lab_rec


def on_subgroups(change):
    """Unticking subgroups on a 3-group setting: collapse to one population whose input from
    each stimulus is the strongest input that stimulus sent to any group, and whose readout /
    recurrent gain are the group averages. Ticking it again copies the population row to all groups."""
    global _hold
    _hold = True
    try:
        M = np.array([[w.value for w in row] for row in mu_in])
        if not change["new"]:
            row, mo, mr = M.max(0), np.mean([w.value for w in mu_out]), np.mean([w.value for w in mu_rec])
            for s in range(3):
                mu_in[0][s].value = float(row[s])
            mu_out[0].value, mu_rec[0].value = float(mo), float(mr)
        else:
            for g in range(1, 3):
                for s in range(3):
                    mu_in[g][s].value = mu_in[0][s].value
                mu_out[g].value, mu_rec[g].value = mu_out[0].value, mu_rec[0].value
            for w in sizes:
                w.value = 1.0
    finally:
        _hold = False
    set_mode(not change["new"])
    redraw()


def load_preset(change=None):
    global _hold
    c = PRESETS[preset.value]
    M, mo, mr = c._arr()
    if c.G == 1:                                   # single population: copy into row 0 (and the rest)
        M, mo, mr = np.repeat(M, 3, 0), np.repeat(mo, 3), np.repeat(mr, 3)
    _hold = True
    try:
        subgroups.unobserve(on_subgroups, names="value")
        subgroups.value = c.G == 3
        subgroups.observe(on_subgroups, names="value")
        for w, x in zip(sizes, np.ones(3) if c.G == 1 else c.sizes):
            w.value = float(x)
        for g in range(3):
            for s in range(3):
                mu_in[g][s].value = float(M[g, s])
            mu_out[g].value, mu_rec[g].value = float(mo[g]), float(mr[g])
        sd_in.value, sd_out.value, sd_rec.value = c.sd_in, c.sd_out, c.sd_rec
        mu_b.value, sd_b.value, readout.value = c.mu_b, c.sd_b, c.readout
    finally:
        _hold = False
    set_mode(c.G == 1)
    redraw()


for w in all_w:
    w.observe(redraw, names="value")
preset.observe(load_preset, names="value")
subgroups.observe(on_subgroups, names="value")

H = widgets.HBox
size_box = widgets.VBox([widgets.HTML("<b>group sizes</b>"), H(sizes)])
ui = widgets.VBox([
    H([preset, subgroups, readout]),
    size_box,
    widgets.HTML("<b>mean input weights</b> (one row per receiving group, or a single row for one population)"),
    *[H(row) for row in mu_in], H([sd_in]),
    widgets.HTML("<b>readout / recurrence / bias</b>"), H(mu_out), H(mu_rec), H([sd_out, sd_rec]), H([mu_b, sd_b]),
    widgets.HTML("<b>simulation</b>"), H([n_units, seed, thresh]),
])
display(ui, out, img)
load_preset()